# Lesson 13: Pretrained language models

Goes with the Lesson 13 page of the Forkcast course. Subword tokenizers, a Hugging Face model as an `nn.Module`, gentle fine-tuning with clipping and accumulation, and sentence embeddings for menu search.

Runtime: pick a **GPU**. The toy model runs anywhere; the real DistilBERT fine-tune wants the GPU (about ten minutes on a T4).

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. `forkcast/lm.py`

The toy tokenizer and toy models are provided. Write `encode_batch`, `ClassificationCollator`, `mean_pool`, `embed_texts`, `search`, `clip_and_step` and `fit_lm` as the page builds them.

In [ ]:
%%writefile forkcast/lm.py
"""Lesson 13. Pretrained language models: a Hugging Face model is an nn.Module, fine-tune it gently, embed sentences.

``train_wordpiece_tokenizer``, ``tiny_distilbert`` and ``tiny_encoder`` are provided so the tests run offline on a
small random model; the notebook uses the real pretrained ones. The rest are exercises.
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import Tensor, nn


def train_wordpiece_tokenizer(texts, vocab_size: int = 1000):
    """A small subword tokenizer trained on our own texts, wrapped in the Hugging Face tokenizer API.

    Real models ship their tokenizer (AutoTokenizer.from_pretrained); this one exists so the tests need no download.
    """
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, processors, trainers
    from transformers import PreTrainedTokenizerFast

    tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    tok.normalizer = normalizers.Sequence([normalizers.Lowercase(), normalizers.NFD(), normalizers.StripAccents()])
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]"])
    tok.train_from_iterator(list(texts), trainer)
    tok.post_processor = processors.TemplateProcessing(single="[CLS] $A [SEP]", special_tokens=[("[CLS]", 2), ("[SEP]", 3)])
    return PreTrainedTokenizerFast(tokenizer_object=tok, pad_token="[PAD]", unk_token="[UNK]", cls_token="[CLS]", sep_token="[SEP]")


def _tiny_config(vocab_size: int, num_labels: int = 2):
    from transformers import DistilBertConfig
    return DistilBertConfig(vocab_size=vocab_size, dim=32, n_layers=1, n_heads=2, hidden_dim=64, max_position_embeddings=128,
                            pad_token_id=0, num_labels=num_labels, dropout=0.0, attention_dropout=0.0)


def tiny_distilbert(vocab_size: int, num_labels: int = 2) -> nn.Module:
    """DistilBERT's architecture at toy size with random weights: the same class the notebook fine-tunes for real."""
    from transformers import DistilBertForSequenceClassification
    return DistilBertForSequenceClassification(_tiny_config(vocab_size, num_labels))


def tiny_encoder(vocab_size: int) -> nn.Module:
    """The encoder alone (no classification head), for sentence embeddings."""
    from transformers import DistilBertModel
    return DistilBertModel(_tiny_config(vocab_size))


def encode_batch(tokenizer, texts, max_length: int = 64) -> dict:
    """Texts -> a dict of tensors the model accepts: input_ids and attention_mask, both (N, T).

    ``tokenizer(list(texts), padding=True, truncation=True, max_length=max_length, return_tensors="pt")``.
    """
    raise NotImplementedError("Lesson 13, step 2: one tokenizer call with padding, truncation, max_length, return_tensors=\"pt\"")


class ClassificationCollator:
    """DataLoader collate_fn for (text, label) pairs: tokenizes the batch and adds a ``labels`` tensor.

    Hugging Face classification models compute their own loss when ``labels`` is in the batch.
    """

    def __init__(self, tokenizer, max_length: int = 64):
        self.tokenizer, self.max_length = tokenizer, max_length

    def __call__(self, batch) -> dict:
        raise NotImplementedError("Lesson 13, step 3: texts, labels = zip(*batch); enc = encode_batch(...); enc[\"labels\"] = torch.tensor(labels); return dict(enc)")


def mean_pool(last_hidden_state: Tensor, attention_mask: Tensor) -> Tensor:
    """(N, T, D), (N, T) -> (N, D): the mean over the real tokens. Lesson 11's masked mean, once more."""
    raise NotImplementedError("Lesson 13, step 4: mask[..., None] weighting, sum over T, divide by the real length")


@torch.no_grad()
def embed_texts(encoder: nn.Module, tokenizer, texts, device: str | torch.device = "cpu", batch_size: int = 32, max_length: int = 64) -> Tensor:
    """Sentence embeddings: mean-pooled encoder outputs, L2-normalised so dot products are cosine similarities. (N, D)

    encoder.eval(); for each batch: encode_batch -> move to device -> encoder(**batch).last_hidden_state -> mean_pool -> F.normalize.
    """
    raise NotImplementedError("Lesson 13, step 4: loop over batches of texts, pool, normalise, torch.cat")


def search(query: str, texts, embeddings: Tensor, encoder: nn.Module, tokenizer, k: int = 3, device: str | torch.device = "cpu") -> list[tuple[str, float]]:
    """The k texts most similar to the query: embed it, dot with the normalised embeddings, take the top k."""
    raise NotImplementedError("Lesson 13, step 4: q = embed_texts(..., [query]); scores = embeddings @ q[0]; topk")


def clip_and_step(model: nn.Module, optimizer: torch.optim.Optimizer, max_grad_norm: float) -> float:
    """Clip the gradient norm, take the optimizer step, zero the gradients. Returns the norm before clipping.

    ``torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)`` rescales every gradient so their
    combined norm is at most max_grad_norm, and returns the norm it found. One oversized step can undo a
    pretrained model; clipping is the seat belt.
    """
    raise NotImplementedError("Lesson 13, step 5: norm = clip_grad_norm_(...); optimizer.step(); optimizer.zero_grad(set_to_none=True); return float(norm)")


def fit_lm(model: nn.Module, train_loader, val_loader, epochs: int = 1, lr: float = 2e-5, accumulation_steps: int = 1,
           max_grad_norm: float = 1.0, device: str | torch.device = "cpu") -> dict:
    """Fine-tuning loop for Hugging Face classification models.

    AdamW. For each batch (a dict): move it to device, ``out = model(**batch)``, ``loss = out.loss / accumulation_steps``,
    backward; every ``accumulation_steps`` batches (and at the end of the epoch) call clip_and_step. Gradient
    accumulation makes several small batches act as one large one, which is how a 6 GB GPU trains a 66M-parameter
    model. After each epoch evaluate the mean validation loss and accuracy (``out.logits.argmax(-1)``).
    Returns {"train": [...], "val": [...], "val_acc": [...], "steps": total optimizer steps}.
    """
    raise NotImplementedError("Lesson 13, step 5: the loop described above")

## 3. Subword pieces, toy and real

In [ ]:
import importlib, math, torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from forkcast import lm, text
from forkcast.model import count_parameters
importlib.reload(lm)

frame = text.load_reviews()
toy_tok = lm.train_wordpiece_tokenizer(frame["text"], vocab_size=600)
real_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
for name, tok in [("toy", toy_tok), ("real", real_tok)]:
    enc = lm.encode_batch(tok, ["Overpriced forkcast pizza, soggy fries!", "Cold."], max_length=32)
    print(name, enc["input_ids"].shape, tok.convert_ids_to_tokens(enc["input_ids"][0].tolist()))
    print("     mask of the short review:", enc["attention_mask"][1].tolist())

## 4. A model is a module

In [ ]:
importlib.reload(lm)
collate = lm.ClassificationCollator(toy_tok, max_length=32)
cut = 640
train_loader = DataLoader(list(zip(frame["text"][:cut], frame["label"][:cut])), batch_size=32, shuffle=True, collate_fn=collate)
val_loader = DataLoader(list(zip(frame["text"][cut:], frame["label"][cut:])), batch_size=64, collate_fn=collate)
batch = next(iter(train_loader)); print({k: tuple(v.shape) for k, v in batch.items()})
torch.manual_seed(0)
tiny = lm.tiny_distilbert(toy_tok.vocab_size, num_labels=2)
out = tiny(**batch)
print(type(tiny).__name__, count_parameters(tiny), "parameters |", out.logits.shape, out.loss)

## 5. Embeddings and search, toy encoder first

In [ ]:
importlib.reload(lm)
menu = ["spicy chicken ramen with soft egg", "margherita pizza with fresh basil", "beef pho with lime and herbs", "green curry with jasmine rice", "crispy fish tacos"]
enc = lm.tiny_encoder(toy_tok.vocab_size)
emb = lm.embed_texts(enc, toy_tok, menu)
print(emb.shape, emb.norm(dim=1))
print(lm.search("spicy chicken ramen with egg", menu, emb, enc, toy_tok, k=3))

## 6. Fine-tune the toy model with clipping and accumulation

In [ ]:
importlib.reload(lm)
torch.manual_seed(0)
tiny = lm.tiny_distilbert(toy_tok.vocab_size, num_labels=2)
h = lm.fit_lm(tiny, train_loader, val_loader, epochs=3, lr=1e-3, accumulation_steps=2)
print("optimizer steps:", h["steps"], "| batches per epoch:", len(train_loader))
print("validation accuracy by epoch:", [round(a, 3) for a in h["val_acc"]])

## 7. The real thing: DistilBERT on star ratings

10,000 Yelp reviews with 1 to 5 stars, one epoch at 2e-5. About ten minutes on a T4.

In [ ]:
!pip install -q datasets
from datasets import load_dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stream = load_dataset("fancyzhx/yelp_review_full", split="train", streaming=True).shuffle(seed=0, buffer_size=10_000)
texts, stars = [], []
for ex in stream:
    texts.append(ex["text"]); stars.append(ex["label"])          # 0..4 = one to five stars
    if len(texts) >= 10_000:
        break
real_collate = lm.ClassificationCollator(real_tok, max_length=128)
r_train = DataLoader(list(zip(texts[1000:], stars[1000:])), batch_size=16, shuffle=True, collate_fn=real_collate)
r_val = DataLoader(list(zip(texts[:1000], stars[:1000])), batch_size=64, collate_fn=real_collate)
torch.manual_seed(0)
bert = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=5)
print(count_parameters(bert), "parameters")
h = lm.fit_lm(bert, r_train, r_val, epochs=1, lr=2e-5, accumulation_steps=2, device=device)
print("star accuracy:", round(h["val_acc"][-1], 3), "(chance 0.20) | optimizer steps:", h["steps"])

bert.eval(); preds = []
with torch.no_grad():
    for b in r_val:
        preds.append(bert(**{k: v.to(device) for k, v in b.items()}).logits.argmax(-1).cpu())
preds = torch.cat(preds); truth = torch.tensor(stars[:1000])
print("mean absolute error in stars:", round((preds - truth).abs().float().mean().item(), 2))

## 8. Search the menu by meaning

A sentence-transformers encoder built for embeddings, against plain DistilBERT.

In [ ]:
menu40 = ["spicy chicken ramen with soft egg", "tonkotsu ramen", "beef pho with lime and herbs", "green curry with jasmine rice", "pad thai with prawns", "margherita pizza with fresh basil", "four cheese lasagna", "spaghetti carbonara", "crispy fish tacos", "classic cheeseburger and fries", "falafel wrap with tahini", "greek salad with feta", "lentil soup", "grilled salmon with asparagus", "mushroom risotto", "chicken tikka masala", "lamb biryani", "vegetable dumplings", "kimchi fried rice", "bibimbap with tofu", "caesar salad", "tomato basil soup", "shrimp pad see ew", "quinoa bowl with roasted vegetables", "fish and chips", "pulled pork sandwich", "eggplant parmigiana", "miso soup with tofu", "chicken shawarma plate", "sushi platter", "poke bowl with tuna", "stuffed peppers", "pepperoni pizza", "beef tacos al pastor", "butter chicken", "udon noodle soup", "caprese sandwich", "mac and cheese", "coconut lentil dal", "chocolate lava cake"]
mini_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
mini = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(device)
plain = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
emb_mini = lm.embed_texts(mini, mini_tok, menu40, device)
emb_plain = lm.embed_texts(plain, real_tok, menu40, device)
print(emb_mini.shape, emb_plain.shape)
for q in ["something spicy with noodles", "a light vegetarian lunch", "comfort food for a cold evening"]:
    print(f"
{q!r}")
    print("  MiniLM:    ", [t for t, s in lm.search(q, menu40, emb_mini, mini, mini_tok, k=3, device=device)])
    print("  DistilBERT:", [t for t, s in lm.search(q, menu40, emb_plain, plain, real_tok, k=3, device=device)])

### Break it on purpose: fine-tune at 1e-3

In [ ]:
torch.manual_seed(0)
reckless = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=5)
h = lm.fit_lm(reckless, r_train, r_val, epochs=1, lr=1e-3, accumulation_steps=2, device=device)
print("at lr=1e-3: star accuracy", round(h["val_acc"][-1], 3), "(2e-5 gave far more)")

## 9. Definition of done

Expected: `7 passed`, then `97 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_13.py -q

In [ ]:
!pytest -m fast -q

## 10. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 13: pretrained language models"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main